# ML Domain-Adaptation Demo — 205 PDFs
**Qwen3-0.6B-Base · NeMo Curator · NeMo AutoModel · One Colab T4**

This notebook performs causal next-token domain-adaptive pretraining on ordinary Machine Learning text from your PDFs. It does not create instruction/answer pairs.

**Workflow:** extract PDFs → document-level train/validation split → NeMo Curator filtering → tokenize → baseline held-out loss → NeMo AutoModel QLoRA → post-training loss → save logs/metrics/artifacts.

**Important:** this notebook is configured for a single T4 and a short faculty demo, not a full production training run. It has not been executed on your Colab account. NeMo APIs and package compatibility can change; if the AutoModel stage fails, retain the full traceback rather than rerunning blindly. No OpenAI API key is used.

## 1. Runtime, paths, and limits

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, sys, json, re, math, random, hashlib, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
import torch

SEED = 42
random.seed(SEED)
WORK = Path("/content/drive/MyDrive/DSU-Finetuning_Project")
RAW, DATA, ART = WORK/"raw", WORK/"data", WORK/"artifacts"
for p in [RAW, DATA, ART]: p.mkdir(parents=True, exist_ok=True)

PDF_DIR = Path("/content/drive/MyDrive/DSU-Finetuning_Project/doc")  # edit to your Google Drive folder
MODEL_ID = "Qwen/Qwen3-0.6B-Base"
SEQ_LEN = 256
MAX_TRAIN_BLOCKS = 6400  # Increased from 120 to allow more training steps
MAX_VAL_BLOCKS = 160
TRAIN_STEPS = 30
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 2e-4

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError("Select Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

Python: 3.13.15 | PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
VRAM GiB: 14.56


## 2. Mount Drive, find PDFs, and install packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
pdf_files = sorted(PDF_DIR.glob("*.pdf")) if PDF_DIR.exists() else []
print("PDF folder:", PDF_DIR, "| PDFs found:", len(pdf_files))
if not pdf_files: raise FileNotFoundError("Place the 205 PDFs in MyDrive/ML_PDFs or edit PDF_DIR.")

%pip -q install -U nemo-automodel nemo-curator pymupdf transformers accelerate bitsandbytes peft pyyaml huggingface_hub

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF folder: /content/drive/MyDrive/DSU-Finetuning_Project/doc | PDFs found: 205
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 MB 9.1 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## 3. Hugging Face authentication (optional for public model)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("No HF_TOKEN secret found. Continuing unauthenticated; public model access may work.")

Hugging Face login complete.


## 4. Extract PDF text and cache each document

In [ ]:
import fitz

records, errors = [], []
for i, path in enumerate(pdf_files, 1):
    cache = RAW / (hashlib.sha1(str(path).encode()).hexdigest() + ".json")
    if cache.exists():
        records.append(json.loads(cache.read_text(encoding="utf-8")))
        continue
    try:
        doc = fitz.open(path)
        text = "\\n".join(page.get_text("text") for page in doc)
        pages = len(doc); doc.close()
        text = re.sub(r"[ \\t]+", " ", text.replace("\\x00", " "))
        text = re.sub(r"\\n{3,}", "\\n\\n", text).strip()
        row = {"doc_id": hashlib.sha1(str(path).encode()).hexdigest()[:16],
               "file_name": path.name, "source_path": str(path), "pages": pages,
               "text": text, "chars": len(text),
               "sha256": hashlib.sha256(text.encode()).hexdigest()}
        cache.write_text(json.dumps(row, ensure_ascii=False), encoding="utf-8")
        records.append(row)
        print(f"{i}/{len(pdf_files)} {path.name}: {pages} pages, {len(text)} chars")
    except Exception as e:
        errors.append({"file": path.name, "error": repr(e)})
(WORK/"extraction_errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")
usable, seen = [], set()
for r in records:
    t = re.sub(r"\\s+", " ", r["text"]).strip()
    h = hashlib.sha256(t.encode()).hexdigest()
    if len(t) >= 500 and h not in seen:
        seen.add(h); usable.append({**r, "text": t, "chars": len(t), "sha256": h})
print("PDFs found:", len(pdf_files), "extracted:", len(records), "usable unique:", len(usable))
print("Low-text PDFs (<500 chars; possibly scanned):", sum(len(r["text"]) < 500 for r in records))
if len(usable) < 5: raise RuntimeError("Too few PDFs have extractable text. Scanned PDFs require OCR.")
print("\\nSample:", usable[0]["file_name"], "\\n", usable[0]["text"][:800])

1/205 3-D Printed Graphene-Based Piezoresistive Foam Mat for Pressure Detection Through Electrical Resistance Tomography and Machine Learning Classification Techniques.pdf: 4 pages, 20166 chars
2/205 A Brute-Force Black-Box Method to Attack Machine Learning-Based Systems in Cybersecurity.pdf: 14 pages, 68552 chars
3/205 A Clinical Decision Support System for Edge-Cloud ICU Readmission Model Based on Particle Swarm Optimization- Ensemble Machine Learning- and Explainable Artificial Intelligence.pdf: 18 pages, 74891 chars
4/205 A Comparative Analysis of Machine Learning and Deep Reinforcement Learning Approaches for Adaptive Intrusion Detection.pdf: 17 pages, 89944 chars
5/205 A Human-Machine Agent Based on Active Reinforcement Learning for Target Classification in Wargame.pdf: 13 pages, 59792 chars
6/205 A Human-Machine Joint Learning Framework to Boost Endogenous BCI Training.pdf: 15 pages, 87259 chars
7/205 A Hybrid Machine Learning Model for Efficient XML Parsing.pdf: 12 pages, 55488

## 5. Split by whole PDF before token chunking

In [ ]:
rng = random.Random(SEED)
shuffled = usable.copy(); rng.shuffle(shuffled)
n_val = max(1, round(0.15 * len(shuffled)))
val_docs, train_docs = shuffled[:n_val], shuffled[n_val:]
manifest = {
  "created_utc": datetime.now(timezone.utc).isoformat(), "model_id": MODEL_ID,
  "split": "whole-PDF split before chunking", "seed": SEED,
  "train": [{"file":r["file_name"],"chars":r["chars"],"sha256":r["sha256"]} for r in train_docs],
  "validation": [{"file":r["file_name"],"chars":r["chars"],"sha256":r["sha256"]} for r in val_docs]
}
(WORK/"corpus_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Train PDFs:", len(train_docs), "| validation PDFs:", len(val_docs))

Train PDFs: 174 | validation PDFs: 31


## 6. Curate the training corpus with NeMo Curator

In [ ]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False) + "\n")

curator_in = DATA/"curator_input.jsonl"
write_jsonl(curator_in, [{"id":r["doc_id"],"text":r["text"],"file_name":r["file_name"]} for r in train_docs])
curator_out = DATA/"curated"
curator_out.mkdir(exist_ok=True)
CURATOR_OK = False
try:
    import nemo_curator
    from nemo_curator.pipeline import Pipeline
    from nemo_curator.backends.xenna import XennaExecutor
    from nemo_curator.stages.text.io.reader import JsonlReader
    from nemo_curator.stages.text.io.writer import JsonlWriter
    from nemo_curator.stages.text.filters import ScoreFilter
    from nemo_curator.stages.text.filters.heuristic import WordCountFilter, NonAlphaNumericFilter

    # Initialize pipeline without arguments
    pipe = Pipeline(name="ml_pdf_curate")

    pipe.add_stage(JsonlReader(file_paths=str(curator_in), fields=["text","id","file_name"]))
    pipe.add_stage(ScoreFilter(filter_obj=WordCountFilter(min_words=80, max_words=100000), text_field="text"))
    pipe.add_stage(ScoreFilter(filter_obj=NonAlphaNumericFilter(max_non_alpha_numeric_to_text_ratio=0.35), text_field="text"))
    pipe.add_stage(JsonlWriter(path=str(curator_out)))

    # Pass XennaExecutor with batch execution mode inside pipe.run()
    executor = XennaExecutor(config={"execution_mode": "batch"})
    pipe.run(executor=executor)

    CURATOR_OK = True
    print("NeMo Curator pipeline completed successfully via XennaExecutor (Batch Mode).")
except Exception as e:
    print("NeMo Curator failed in this runtime:", type(e).__name__, str(e)[:1000])
    print("Transparent local heuristic fallback will be used; this run will record CURATOR_OK=False.")

curated = []
if CURATOR_OK:
    for fp in curator_out.rglob("*.jsonl"):
        for line in fp.read_text(encoding="utf-8").splitlines():
            try: curated.append(json.loads(line))
            except Exception: pass
if not curated:
    for r in train_docs:
        words = r["text"].split()
        nonalpha = sum(not ch.isalnum() and not ch.isspace() for ch in r["text"]) / max(1,len(r["text"]))
        if len(words) >= 80 and nonalpha <= 0.35:
            curated.append({"id":r["doc_id"],"text":r["text"],"file_name":r["file_name"]})
curated = [r for r in curated if isinstance(r.get("text"),str) and len(r.get("text","").split()) >= 80]
print("Curated documents:", len(curated), "| NeMo Curator actually ran:", CURATOR_OK)
if len(curated) < 3: raise RuntimeError("Too little text after filtering; inspect PDF extraction/filter thresholds.")

2026-10-12 06:14:31.532 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'jsonl_reader' to pipeline 'ml_pdf_curate'
2026-10-12 06:14:31.541 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'word_count' to pipeline 'ml_pdf_curate'
2026-10-12 06:14:31.544 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'alpha_numeric' to pipeline 'ml_pdf_curate'
2026-10-12 06:14:31.548 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'jsonl_writer' to pipeline 'ml_pdf_curate'
2026-10-12 06:14:31.551 | INFO     | nemo_curator.pipeline.pipeline:build:123 - Planning pipeline: ml_pdf_curate
2026-10-12 06:14:31.556 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:184 - Decomposing composite stage: jsonl_reader
2026-10-12 06:14:31.557 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:198 - Expanded 'jsonl_reader' into 2 execution stages
2026-10-12 06:14:31.561 | INFO     | nemo_curator.backends

NeMo Curator pipeline completed successfully via XennaExecutor (Batch Mode).
Curated documents: 174 | NeMo Curator actually ran: True


## 7. Tokenize into causal next-token blocks

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, token=HF_TOKEN or None)
if not tokenizer.eos_token: raise RuntimeError("No EOS token found.")

def to_blocks(rows, limit):
    blocks = []
    for row in rows:
        ids = tokenizer(row["text"].strip()+tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        for start in range(0, len(ids)-SEQ_LEN+1, SEQ_LEN):
            ids_part = ids[start:start+SEQ_LEN]
            if len(ids_part) == SEQ_LEN:
                blocks.append({"input_ids":ids_part,"attention_mask":[1]*SEQ_LEN,"labels":ids_part.copy()})
                if len(blocks) >= limit: return blocks
    return blocks

train_blocks = to_blocks(curated, MAX_TRAIN_BLOCKS)
val_blocks = to_blocks([{"text":r["text"]} for r in val_docs], MAX_VAL_BLOCKS)
write_jsonl(DATA/"train_blocks.jsonl", train_blocks)
write_jsonl(DATA/"validation_blocks.jsonl", val_blocks)
print("Train blocks:",len(train_blocks),"| validation blocks:",len(val_blocks),"| tokens/block:",SEQ_LEN)

Train blocks: 6400 | validation blocks: 160 | tokens/block: 256


## 8. Baseline held-out loss

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

def load_eval_model():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID, token=HF_TOKEN or None, quantization_config=bnb,
        device_map="auto", torch_dtype=torch.float16, low_cpu_mem_usage=True)

def evaluate_loss(model, blocks, max_batches=40):
    model.eval(); losses=[]
    for r in blocks[:max_batches]:
        batch = {k:torch.tensor([r[k]], dtype=torch.long, device=model.device)
                 for k in ("input_ids","attention_mask","labels")}
        with torch.no_grad(): out=model(**batch)
        if torch.isfinite(out.loss): losses.append(float(out.loss.detach().cpu()))
    return sum(losses)/len(losses), len(losses)

base_model = load_eval_model()
baseline_loss, baseline_n = evaluate_loss(base_model, val_blocks)
print(f"BASELINE held-out loss: {baseline_loss:.4f} ({baseline_n} blocks)")
del base_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

BASELINE held-out loss: 3.8014 (40 blocks)


## 9. Create NeMo AutoModel dataset factory and QLoRA config

In [ ]:
# ==========================================
# Optimized NeMo AutoModel Configuration (Safe for T4 16GB VRAM)
# ==========================================
import sys, yaml

module_file = WORK/"ml_dapt_dataset.py"
module_file.write_text('''
import json, torch
from torch.utils.data import Dataset
class CausalBlocks(Dataset):
    def __init__(self, jsonl_path, limit_samples=None):
        self.rows=[]
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.rows.append(json.loads(line))
                    if limit_samples and len(self.rows)>=int(limit_samples): break
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r=self.rows[idx]
        return {k:torch.tensor(r[k], dtype=torch.long) for k in ("input_ids","attention_mask","labels")}
def make_causal_dataset(jsonl_path, limit_samples=None):
    return CausalBlocks(jsonl_path, limit_samples)
''', encoding="utf-8")

sys.path.insert(0, str(WORK))

TOTAL_TRAIN_STEPS = 200

cfg = {
 "recipe": "TrainFinetuneRecipeForNextTokenPrediction",
 "step_scheduler": {
     "ckpt_every_steps": 50,
     "val_every_steps": 50,
     "max_steps": TOTAL_TRAIN_STEPS,
     "num_epochs": 1
 },
 "dist_env": {"backend": "nccl", "timeout_minutes": 10},
 "distributed": {"strategy": "ddp", "dp_size": 1},
 "rng": {"_target_": "nemo_automodel.components.training.rng.StatefulRNG", "seed": SEED, "ranked": True},
 "model": {
      "_target_": "nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained",
      "pretrained_model_name_or_path": MODEL_ID,
      "torch_dtype": "float32",  # <--- Changed from float32 to float16 to prevent OOM
 },
 "peft": {
      "_target_": "nemo_automodel.components._peft.lora.PeftConfig",
      "target_modules": "*_proj",
      "dim": 8,
      "alpha": 16,
      "dropout": 0.05,
      "use_triton": False,
 },
 "quantization": {
      "load_in_4bit": False,
      "load_in_8bit": False,
      "bnb_4bit_compute_dtype": "float16",
      "bnb_4bit_use_double_quant": True,
      "bnb_4bit_quant_type": "nf4",
      "bnb_4bit_quant_storage": "float16",
 },
 "dataset": {
      "_target_": "ml_dapt_dataset.make_causal_dataset",
      "jsonl_path": str(DATA / "train_blocks.jsonl"),
      "limit_samples": MAX_TRAIN_BLOCKS,
      # limit_samples omitted to process all blocks safely in float16
 },
 "validation_dataset": {
      "_target_": "ml_dapt_dataset.make_causal_dataset",
      "jsonl_path": str(DATA / "validation_blocks.jsonl"),
      "limit_samples": MAX_TRAIN_BLOCKS,
 },
 "dataloader": {
      "_target_": "torchdata.stateful_dataloader.StatefulDataLoader",
      "collate_fn": "nemo_automodel.components.datasets.utils.default_collater",
      "batch_size": BATCH_SIZE,
      "shuffle": True,
 },
 "validation_dataloader": {
      "_target_": "torchdata.stateful_dataloader.StatefulDataLoader",
      "collate_fn": "nemo_automodel.components.datasets.utils.default_collater",
      "batch_size": 1,
 },
 "loss_fn": {"_target_": "nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy"},
 "optimizer": {
      "_target_": "torch.optim.AdamW",
      "lr": 1e-5,
      "betas": [0.9, 0.999],
      "eps": 1e-8,
      "weight_decay": 0.0,
 },
 "checkpoint": {
      "enabled": True,
      "checkpoint_dir": str(ART / "nemo_checkpoints"),
      "model_save_format": "safetensors",
      "save_consolidated": "final",
 },
}

config_path = WORK / "nemo_dapt_config.yaml"
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(config_path.read_text())

recipe: TrainFinetuneRecipeForNextTokenPrediction
step_scheduler:
  ckpt_every_steps: 50
  val_every_steps: 50
  max_steps: 200
  num_epochs: 1
dist_env:
  backend: nccl
  timeout_minutes: 10
distributed:
  strategy: ddp
  dp_size: 1
rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 42
  ranked: true
model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: Qwen/Qwen3-0.6B-Base
  torch_dtype: float32
peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  target_modules: '*_proj'
  dim: 8
  alpha: 16
  dropout: 0.05
  use_triton: false
quantization:
  load_in_4bit: false
  load_in_8bit: false
  bnb_4bit_compute_dtype: float16
  bnb_4bit_use_double_quant: true
  bnb_4bit_quant_type: nf4
  bnb_4bit_quant_storage: float16
dataset:
  _target_: ml_dapt_dataset.make_causal_dataset
  jsonl_path: /content/drive/MyDrive/DSU-Finetuning_Project/data/train_blocks.jsonl
  limit_samples: 6400
validation_dataset:


## 10. Run NeMo AutoModel training (single GPU)

In [ ]:
#import shutil

# Clear old checkpoints to prevent loading NaN weights
#checkpoint_dir = ART / "nemo_checkpoints"
#if checkpoint_dir.exists():
#    shutil.rmtree(checkpoint_dir)
#    print("Cleared old checkpoint directory.")

#print("Ready to re-run training cell with float32 stability and safe learning rate!")

Cleared old checkpoint directory.
Ready to re-run training cell with float32 stability and safe learning rate!


**Validation loss and perplexity before training the model.**

In [ ]:
import torch
import math
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import json

# 1. Configuration & Paths
WORK = Path("/content/drive/MyDrive/DSU-Finetuning_Project")
DATA = WORK / "data"
ART = WORK / "artifacts"
MODEL_ID = "Qwen/Qwen3-0.6B-Base"  # Or your merged checkpoint path after training

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading tokenizer and model for evaluation on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load base model (or load your trained adapter checkpoint here)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()

# 2. Load Validation Blocks
val_jsonl = DATA / "validation_blocks.jsonl"
val_data = []
if val_jsonl.exists():
    with open(val_jsonl, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                val_data.append(json.loads(line))

print(f"Loaded {len(val_data)} validation blocks for evaluation.")

# 3. Compute Cross-Entropy Loss & Perplexity
total_loss = 0.0
total_tokens = 0

loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")

with torch.no_grad():
    for item in val_data:
        input_ids = torch.tensor([item["input_ids"]], dtype=torch.long).to(device)
        attention_mask = torch.tensor([item["attention_mask"]], dtype=torch.long).to(device)
        labels = torch.tensor([item["labels"]], dtype=torch.long).to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

        # Shift logits and labels for causal next-token prediction
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        valid_tokens = (shift_labels != -100).sum().item()
        total_loss += loss.item()
        total_tokens += valid_tokens

# 4. Calculate Final Metrics
mean_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
perplexity = math.exp(mean_loss) if mean_loss < 100 else float('inf')

print("=" * 40)
print(f"Validation Loss: {mean_loss:.4f}")
print(f"Validation Perplexity: {perplexity:.4f}")
print("=" * 40)

Loading tokenizer and model for evaluation on cuda...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loaded 160 validation blocks for evaluation.
Validation Loss: 3.6659
Validation Perplexity: 39.0930


In [ ]:
# Run training with LIVE output (the old version captured output and printed nothing until the process exited).
import shutil, subprocess, os, re, yaml
from math import ceil
from pathlib import Path

def verify_training(ckpt_dir, total_steps):
    # Exit code 0 does not prove training ran: confirm a checkpoint at (or next to) the final step exists.
    steps = [int(m.group(1)) for d in Path(ckpt_dir).glob("*step_*") if (m := re.search(r"step_(\d+)", d.name))]
    last = max(steps, default=-1)
    if last >= total_steps - 1:
        print(f"\nTraining complete. Latest checkpoint: step_{last} (target {total_steps} steps).")
    else:
        raise RuntimeError(f"Process exited cleanly but the latest checkpoint is step_{last}, short of {total_steps}. "
                           "Check num_epochs / dataset size: max_steps alone cannot extend a run past the last epoch.")

with open(config_path, "r", encoding="utf-8") as f:
    cfg_data = yaml.safe_load(f)

# Fix 1: no sequence_parallel with DDP
if cfg_data.get("distributed", {}).get("strategy") == "ddp":
    cfg_data["distributed"].pop("sequence_parallel", None)

# Fix 2: force-disable 4-bit / 8-bit quantization (bitsandbytes AssertionError)
if "quantization" in cfg_data:
    cfg_data["quantization"]["load_in_4bit"] = False
    cfg_data["quantization"]["load_in_8bit"] = False
    print("Fixed config: Force-disabled 4-bit and 8-bit quantization.")

# Fix 3: max_steps is only a cap. The run also ends when num_epochs ends (default global batch = 32 samples/step),
# so with num_epochs=1 and ~1000 blocks it stops after ~32 steps. Make num_epochs cover max_steps.
GRAD_ACC = 32
TOTAL_TRAIN_STEPS = cfg_data["step_scheduler"].get("max_steps", 200)
n_blocks = cfg_data["dataset"].get("limit_samples") or sum(1 for _ in open(cfg_data["dataset"]["jsonl_path"], encoding="utf-8"))
steps_per_epoch = ceil(n_blocks / GRAD_ACC)
cfg_data["step_scheduler"]["num_epochs"] = ceil(TOTAL_TRAIN_STEPS / steps_per_epoch) + 1   # +1 harmless: max_steps stops the run
print(f"{n_blocks} blocks -> {steps_per_epoch} steps/epoch -> num_epochs={cfg_data['step_scheduler']['num_epochs']} (max_steps={TOTAL_TRAIN_STEPS})")

with open(config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg_data, f, sort_keys=False)

cli = shutil.which("automodel")
if not cli:
    raise RuntimeError("`automodel` CLI not found after install. Inspect pip install output and NeMo AutoModel docs.")
env = os.environ.copy()
env["PYTHONPATH"] = str(WORK) + os.pathsep + env.get("PYTHONPATH", "")
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"
cmd = [cli, "--nproc-per-node=1", str(config_path)]
print("Running:", " ".join(cmd))
print("Model load takes ~1 min; after that you should see a 'Training: x/N [..step/s]' bar and a loss line per step.\n")

proc = subprocess.Popen(cmd, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with open(WORK / "training_stdout.txt", "w", encoding="utf-8") as log:
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
proc.wait()
print("\nExit code:", proc.returncode)
if proc.returncode:
    raise RuntimeError("NeMo AutoModel training failed. Preserve the traceback above; do not rerun blindly if the error is OOM.")
verify_training(cfg_data["checkpoint"]["checkpoint_dir"], TOTAL_TRAIN_STEPS)


In [ ]:
import sys
import os
import yaml
from pathlib import Path
import subprocess

WORK = Path("/content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning")
DATA = WORK / "data"
ART = WORK / "artifacts"
CHECKPOINT_DIR = ART / "nemo_checkpoints"

# Debug step: Verify files in WORK folder to check for ml_dapt_dataset.py
print(f"Checking files in WORK directory ({WORK}):")
if WORK.exists():
    files = [item.name for item in WORK.iterdir()]
    for name in files:
        print(f" - {name}")
    if "ml_dapt_dataset.py" not in files:
        print("\nWARNING: 'ml_dapt_dataset.py' was not found in WORK directory! Please ensure it is uploaded to this folder.")
else:
    print("WARNING: WORK directory does not exist yet!")

# Read your existing config
cfg_file = WORK / "nemo_dapt_config.yaml"
cfg = yaml.safe_load(cfg_file.read_text())

# Ensure max_steps is 200
cfg["step_scheduler"]["max_steps"] = 200

# Dynamically fix the training dataset path to point to the correct path in Google Drive
if "dataset" in cfg and "jsonl_path" in cfg["dataset"]:
    old_path = cfg["dataset"]["jsonl_path"]
    new_path = str(DATA / "train_blocks.jsonl")
    if old_path != new_path:
        print(f"\nCorrecting training dataset jsonl_path from:\n  {old_path}\nto:\n  {new_path}")
        cfg["dataset"]["jsonl_path"] = new_path

# Dynamically fix the validation dataset path to point to the correct path in Google Drive
if "validation_dataset" in cfg and "jsonl_path" in cfg["validation_dataset"]:
    old_val_path = cfg["validation_dataset"]["jsonl_path"]
    new_val_path = str(DATA / "validation_blocks.jsonl")
    if old_val_path != new_val_path:
        print(f"\nCorrecting validation dataset jsonl_path from:\n  {old_val_path}\nto:\n  {new_val_path}")
        cfg["validation_dataset"]["jsonl_path"] = new_val_path

# Dynamically fix the checkpoint directory path
if "checkpoint" in cfg and "checkpoint_dir" in cfg["checkpoint"]:
    old_ckpt_dir = cfg["checkpoint"]["checkpoint_dir"]
    new_ckpt_dir = str(CHECKPOINT_DIR)
    if old_ckpt_dir != new_ckpt_dir:
        print(f"\nCorrecting checkpoint_dir from:\n  {old_ckpt_dir}\nto:\n  {new_ckpt_dir}")
        cfg["checkpoint"]["checkpoint_dir"] = new_ckpt_dir

# Write back
cfg_file.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Config verified. Resuming training to reach the full 200 steps...")

# Execute training stream
env = os.environ.copy()
env["PYTHONPATH"] = str(WORK) + os.pathsep + env.get("PYTHONPATH", "")

cmd = ["/usr/local/bin/automodel", "--nproc-per-node=1", str(cfg_file)]

# Run subprocess with cwd=str(WORK) so Python looks inside WORK directory for modules
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=str(WORK)
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Training continuation failed with exit code {process.returncode}")
else:
    print("\nTraining successfully pushed through to 200 steps!")

Checking files in WORK directory (/content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning):
 - .ipynb_checkpoints
 - artifacts
 - data
 - nemo_dapt_config.yaml
 - training_stdout.txt
 - ml_dapt_dataset.py
 - __pycache__

Correcting validation dataset jsonl_path from:
  /content/drive/MyDrive/DSU-Finetuning_Project/data/validation_blocks.jsonl
to:
  /content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning/data/validation_blocks.jsonl

Correcting checkpoint_dir from:
  /content/drive/MyDrive/DSU-Finetuning_Project/artifacts/nemo_checkpoints
to:
  /content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning/artifacts/nemo_checkpoints
Config verified. Resuming training to reach the full 200 steps...
/usr/local/lib/python3.13/dist-packages/megatron_fsdp/mixed_precision.py:115: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
INFO:nemo_automodel.components.lau

**Validation loss and perplexity after training the model**

In [ ]:
from nemo_automodel import NeMoAutoModelForCausalLM
import torch
import math
import json
import re
from pathlib import Path
from transformers import AutoTokenizer

WORK = Path("/content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning")
DATA = WORK / "data"
ART = WORK / "artifacts"
MODEL_ID = "Qwen/Qwen3-0.6B-Base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer and finding adapter...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

adapter_configs = list(ART.rglob("adapter_config.json"))
if not adapter_configs:
    raise RuntimeError("No saved PEFT adapter config found! Complete training first.")

def get_step_num(path):
    match = re.search(r"step_(\d+)", str(path))
    return int(match.group(1)) if match else -1

adapter_configs = sorted(adapter_configs, key=get_step_num)
adapter_dir = adapter_configs[-1].parent
print(f"Loading adapted model via NeMoAutoModel from: {adapter_dir}")

# Reconstruct with proper namespace wrapping
adapted_model = NeMoAutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    peft_model_id=str(adapter_dir),
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
adapted_model.eval()

# Load Validation Blocks
val_jsonl = DATA / "validation_blocks.jsonl"
val_data = []
if val_jsonl.exists():
    with open(val_jsonl, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                val_data.append(json.loads(line))

# Compute Loss & Perplexity
total_loss = 0.0
total_tokens = 0
loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")

with torch.no_grad():
    for item in val_data:
        input_ids = torch.tensor([item["input_ids"]], dtype=torch.long).to(device)
        attention_mask = torch.tensor([item["attention_mask"]], dtype=torch.long).to(device)
        labels = torch.tensor([item["labels"]], dtype=torch.long).to(device)

        outputs = adapted_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        valid_tokens = (shift_labels != -100).sum().item()

        total_loss += loss.item()
        total_tokens += valid_tokens

mean_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
perplexity = math.exp(mean_loss) if mean_loss < 100 else float('inf')

print("=" * 40)
print(f"ADAPTED Validation Loss: {mean_loss:.4f}")
print(f"ADAPTED Validation Perplexity: {perplexity:.4f}")
print("=" * 40)

[ERROR] `loss` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.
[ERROR] `logits` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.
[ERROR] `loss` is part of Qwen2VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/modeling_qwen2_vl.py.
[ERROR] `logits` is part of Qwen2VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/modeling_qwen2_vl.py.
gr

/usr/local/lib/python3.13/dist-packages/megatron_fsdp/mixed_precision.py:115: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(


Loading tokenizer and finding adapter...
Loading adapted model via NeMoAutoModel from: /content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning/artifacts/nemo_checkpoints/epoch_0_step_199/model


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


ADAPTED Validation Loss: 3.6659
ADAPTED Validation Perplexity: 39.0920


**Side-by-Side Comparison of base model and finetuned model**

In [ ]:
import torch
import sys
import gc
import re
from transformers import AutoTokenizer
from nemo_automodel import NeMoAutoModelForCausalLM

# 0. Clean VRAM
gc.collect()
torch.cuda.empty_cache()

EVAL_MODEL_ID = "Qwen/Qwen3-0.6B-Base"
adapter_configs = list(ART.rglob("adapter_config.json"))

if not adapter_configs:
    raise RuntimeError("No saved PEFT adapter config found! Complete training first.")

def get_step_num(path):
    match = re.search(r"step_(\d+)", str(path))
    return int(match.group(1)) if match else -1

adapter_configs = sorted(adapter_configs, key=get_step_num)
adapter_dir = adapter_configs[-1].parent
print(f"Loading adapter from: {adapter_dir}\n")

sample_prompt = "In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating"
print(f"Domain Test Prompt:\n\"{sample_prompt}\"\n" + "="*60)

tokenizer = AutoTokenizer.from_pretrained(EVAL_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(sample_prompt, return_tensors="pt").to("cuda")

# ------------------------------------------
# 1. Baseline Model
# ------------------------------------------
print("Generating response from Baseline Model...")
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    EVAL_MODEL_ID,
    torch_dtype=torch.float16
).to("cuda")

with torch.no_grad():
    base_outputs = base_model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )
base_text = tokenizer.decode(base_outputs[0], skip_special_tokens=True)

del base_model
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------
# 2. Adapted Model via NeMoAutoModel (Resolves structural wrapper mismatch)
# ------------------------------------------
print("Generating response from Fine-Tuned IEEE ML Model...")
# Loading through NeMoAutoModel correctly maps the base_model.model weights
adapted_nemo = NeMoAutoModelForCausalLM.from_pretrained(
    EVAL_MODEL_ID,
    peft_model_id=str(adapter_dir),
    torch_dtype=torch.float16
).to("cuda")

with torch.no_grad():
    adapted_outputs = adapted_nemo.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )
adapted_text = tokenizer.decode(adapted_outputs[0], skip_special_tokens=True)

del adapted_nemo
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------
# Results
# ------------------------------------------
print("\n" + "="*20 + " [BASELINE MODEL OUTPUT] " + "="*20)
print(base_text)

print("\n" + "="*20 + " [FINETUNED IEEE ML MODEL OUTPUT] " + "="*20)
print(adapted_text)

Loading adapter from: /content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning/artifacts/nemo_checkpoints/epoch_0_step_199/model

Domain Test Prompt:
"In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating"
Generating response from Baseline Model...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Generating response from Fine-Tuned IEEE ML Model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


==================== [BASELINE MODEL OUTPUT] ====================
In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating both memory and contextual information. The model is made up of a transformer encoder that encodes the input and its corresponding context, and a transformer decoder that generates the output. The transformer encoder is followed by a multi-head attention layer that allows the model to capture long-term dependencies in the input. The transformer decoder is followed by a self-attention layer that enables the model to maintain the memory of the input. We have conducted extensive experiments on several benchmark datasets, and we have shown that our model can outperform existing transformer-based models in terms of model accuracy and efficiency. Additionally, we have investigated the effectiveness

==================== [FINETUNED IEEE ML MODEL OUTPUT] ====================
In this paper, we propose a 

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from nemo_automodel import NeMoAutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "Qwen/Qwen3-0.6B-Base"
adapter_dir = "/content/drive/MyDrive/DSU-FineTuning/Non-Instruction Finetuning/artifacts/nemo_checkpoints/epoch_0_step_199/model"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 1. Load Pure Base Model (Unadapted)
print("Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
base_model.eval()

# 2. Load Fine-Tuned Domain Model (via Native NeMo Wrapper)
print("Loading Domain-Adapted Fine-Tuned Model...")
adapted_model = NeMoAutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    peft_model_id=adapter_dir,
    torch_dtype=torch.float16
).to(device)
adapted_model.eval()

# 3. Academic Domain Prompt (Non-Instruction Style)
test_prompt = "In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

# Generation settings for clean output
gen_kwargs = {
    "max_new_tokens": 60,
    "temperature": 0.4,
    "top_p": 0.9,
    "repetition_penalty": 1.4,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id
}

print("\n" + "="*70)
print(f"PROMPT: \"{test_prompt}\"")
print("="*70)

# Generate Base Output
with torch.no_grad():
    base_outputs = base_model.generate(**inputs, **gen_kwargs)
base_text = tokenizer.decode(base_outputs[0], skip_special_tokens=True)

# Generate Adapted Output
with torch.no_grad():
    adapted_outputs = adapted_model.generate(**inputs, **gen_kwargs)
adapted_text = tokenizer.decode(adapted_outputs[0], skip_special_tokens=True)

print("\n--- 1. BASE MODEL OUTPUT (Before Fine-Tuning) ---")
print(base_text)

print("\n--- 2. DOMAIN-ADAPTED OUTPUT (After Non-Instruction DAPT) ---")
print(adapted_text)
print("="*70)

Loading Tokenizer...
Loading Base Model...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading Domain-Adapted Fine-Tuned Model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


PROMPT: "In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating"

--- 1. BASE MODEL OUTPUT (Before Fine-Tuning) ---
In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating the concept of "memory" in its design. Our approach is inspired from deep learning models that use memory as an essential component for long-term data retention and retrieval.
The proposed model incorporates several key components:
1) Memory: A self-attention mechanism with attention weights calculated based on past information (i

--- 2. DOMAIN-ADAPTED OUTPUT (After Non-Instruction DAPT) ---
In this paper, we propose a novel transformer-based architecture designed to mitigate catastrophic forgetting by incorporating an attention mechanism. We introduce the concept of "attention-aware memory", where each element in our model is assigned weights based on its relevance during t